# Three-seed LIF audit: latest/best, ammo efficiency and replay

Evaluation first: use your existing seed-0/1/2 experiments. No retraining by default.
Each checkpoint is handled in a fresh subprocess so model registrations and CUDA memory
are isolated. Latest and best are reported separately; best means best recorded training
reward, not best held-out performance. The current default names match the reported runs.
Existing training notebooks are not modified. Requires a Linux GPU allocation and ffmpeg
with libx264 for videos. Jobs execute sequentially. Do not run while checkpoints are being written.


In [ ]:
import os, sys, json, random, hashlib, time, importlib
from pathlib import Path
from importlib.metadata import version
# This variable also reaches newly spawned Sample Factory checkpoint loaders.
# Only load checkpoints produced by this project or another trusted source.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
import numpy as np
import torch
import gymnasium as gym
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "defend the center").is_dir() and (p / "conversion_friendly_v1").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from this repository.")
WORK_DIR = ROOT / "defend the center"
os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path: sys.path.insert(0, str(WORK_DIR))
OUTPUT_ROOT = Path(os.environ.get("DEFEND_OUTPUT_ROOT", str(WORK_DIR / "outputs"))).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(1)
assert torch.cuda.is_available(), "Launch this kernel inside a GPU allocation."
DEVICE = "cuda"
CPU_BUDGET = int(os.environ.get("SLURM_CPUS_PER_TASK", len(os.sched_getaffinity(0))))
NUM_WORKERS = max(1, min(8, CPU_BUDGET, int(os.environ.get("DOOM_NUM_WORKERS", max(1, CPU_BUDGET-2)))))
print("Kernel:", sys.executable, "GPU:", torch.cuda.get_device_name(0), "workers:", NUM_WORKERS)
print("GPU test:", (torch.ones(1, device=DEVICE)+1).item())


## Settings

Adjust experiment names if your runs use another tag/output root. Keep all evaluation seeds identical. Use a new audit tag for a fresh measurement; completed jobs in the same audit are reused. Failed/incomplete jobs require an explicit retry.

In [ ]:
TRAINING_SEEDS = [0,1,2]
EXPERIMENTS = {seed:f"defend_lif_beta0.9_T8_seed{seed}_v1" for seed in TRAINING_SEEDS}
CHECKPOINT_KINDS = ["latest","best"]
EVAL_SEEDS = list(range(10000,10300))
VIDEO_SEEDS = [20000,20001,20002]  # demonstration seeds, not final untouched test data
AUDIT_TAG = "lif_seeds012_v1"
RETRY_INCOMPLETE = False
TRAINING_ACTION = "none"  # none, fresh, resume -- training is opt-in
TOTAL_TRAINING_STEPS = 8_000_000
assert TRAINING_ACTION in {"none","fresh","resume"}
AUDIT_DIR = OUTPUT_ROOT / "seed_audits" / AUDIT_TAG
AUDIT_DIR.mkdir(parents=True,exist_ok=True)


## Prepare isolated workers

Writes an importable copy of the existing LIF implementation into this audit directory. The worker uses saved run configurations and original gains for resume/evaluation. Fresh mode calibrates new gains and refuses existing experiment directories. Fixed fresh defaults match beta=0.9, T=8, LR=1e-4, batch=1024. Training targets for resumed runs must be checked in their resolved logs/config.

In [ ]:
import subprocess, shutil, threading, queue
from IPython.display import Video, display
MODULE_SOURCE = '"""Directly trained spiking policy for Sample Factory (ViZDoom doom_defend_the_center).\n\nSame layer sizes as the ANN baseline (convnet_simple): 3 convs + a 512 layer, but every\nhidden layer is an leaky integrate-and-fire (LIF) layer unrolled over T time steps and trained\nwith surrogate gradients. State is reset for every frame (no recurrence across frames).\n"""\nimport numpy as np\nimport torch\nfrom torch import nn\n\nfrom sample_factory.algo.utils.torch_utils import calc_num_elements\nfrom sample_factory.model.encoder import Encoder\n\n\n# ----------------------------------------------------------------------------------------\n# Neuron: Heaviside spike in the forward pass, triangular surrogate in the backward pass\n# ----------------------------------------------------------------------------------------\nclass SpikeFn(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, x):  # x = membrane - threshold\n        ctx.save_for_backward(x)\n        return (x >= 0).to(x.dtype)\n\n    @staticmethod\n    def backward(ctx, grad_out):\n        (x,) = ctx.saved_tensors\n        return grad_out * torch.clamp(1.0 - x.abs(), min=0.0)  # triangular, width 1\n\n\ndef lif_step(v, current, threshold=1.0, v_init=0.5, detach_reset=True, beta=0.9):\n    """One LIF step: decay voltage, add current, spike, subtract threshold.\n\n    beta=1 recovers the IF update; current is not multiplied by (1-beta)."""\n    if v is None:\n        v = torch.full_like(current, v_init * threshold)\n    v = beta * v + current\n    s = SpikeFn.apply(v - threshold)\n    v = v - (s.detach() if detach_reset else s) * threshold\n    return v, s\n\n\n# ----------------------------------------------------------------------------------------\n# The spiking trunk (no Sample Factory dependency in the maths, so it can be evaluated alone)\n# ----------------------------------------------------------------------------------------\nclass SpikingTrunk(nn.Module):\n    def __init__(self, in_channels=3, obs_hw=(72, 128), T=8, v_init=0.5,\n                 gains=(1.0, 1.0, 1.0, 1.0), feat_scale=4.0, beta=0.9):\n        super().__init__()\n        if not 0.0 <= float(beta) <= 1.0:\n            raise ValueError("beta must be in [0, 1]")\n        self.register_buffer("beta", torch.tensor(float(beta)))\n        self.T = int(T)\n        self.v_init = float(v_init)\n        self.conv1 = nn.Conv2d(in_channels, 32, 8, stride=4)\n        self.conv2 = nn.Conv2d(32, 64, 4, stride=2)\n        self.conv3 = nn.Conv2d(64, 128, 3, stride=2)\n        # local (unregistered) stack, only used to compute the flattened size; keeps state_dict keys unique\n        self.flat_size = calc_num_elements(nn.Sequential(self.conv1, self.conv2, self.conv3), (in_channels, *obs_hw))\n        self.fc = nn.Linear(self.flat_size, 512)\n        # fixed (non-trainable) per-layer current gains and output scale: buffers, so they are saved\n        # in the checkpoint and are not touched by Sample Factory\'s weight initialisation\n        self.register_buffer("gains", torch.tensor(list(gains), dtype=torch.float32))\n        self.register_buffer("feat_scale", torch.tensor(float(feat_scale)))\n        self.out_size = 512\n\n    def forward(self, x, T=None, return_spikes=False):\n        T = self.T if T is None else int(T)\n        g = self.gains\n        c1 = self.conv1(x) * g[0]  # direct encoding: identical every step, computed once\n        v1 = v2 = v3 = v4 = None\n        acc = 0\n        spikes = [0.0, 0.0, 0.0, 0.0]\n        for _ in range(T):\n            v1, s1 = lif_step(v1, c1, v_init=self.v_init, beta=self.beta)\n            v2, s2 = lif_step(v2, self.conv2(s1) * g[1], v_init=self.v_init, beta=self.beta)\n            v3, s3 = lif_step(v3, self.conv3(s2) * g[2], v_init=self.v_init, beta=self.beta)\n            v4, s4 = lif_step(v4, self.fc(s3.flatten(1)) * g[3], v_init=self.v_init, beta=self.beta)\n            acc = acc + s4\n            if return_spikes:\n                for k, s in enumerate((s1, s2, s3, s4)):\n                    spikes[k] = spikes[k] + s.detach().flatten(1).sum(1)  # per-sample spike count\n        feats = acc * (self.feat_scale / T)  # mean firing rate of the 512 layer, scaled\n        if return_spikes:\n            return feats, torch.stack(spikes, 1)  # (B, 4) spikes per decision per layer\n        return feats\n\n\n# ----------------------------------------------------------------------------------------\n# Sample Factory glue\n# ----------------------------------------------------------------------------------------\ndef add_snn_args(parser):\n    p = parser\n    p.add_argument("--snn_beta", default=0.9, type=float, help="Fixed LIF membrane retention in [0, 1]")\n    p.add_argument("--snn_T", default=8, type=int, help="Time steps per frame for the spiking encoder")\n    p.add_argument("--snn_v_init", default=0.5, type=float, help="Initial membrane potential / threshold")\n    p.add_argument("--snn_gains", default="1,1,1,1", type=str,\n                   help="Comma separated fixed current gains for the 4 spiking layers (from calibration)")\n    p.add_argument("--snn_feat_scale", default=4.0, type=float, help="Scale on the mean firing rate fed to the heads")\n\n\nclass SpikingEncoder(Encoder):\n    def __init__(self, cfg, obs_space):\n        super().__init__(cfg)\n        obs_shape = obs_space["obs"].shape\n        gains = [float(v) for v in str(cfg.snn_gains).split(",")]\n        assert len(gains) == 4, "--snn_gains needs 4 comma separated numbers"\n        self.trunk = SpikingTrunk(obs_shape[0], tuple(obs_shape[1:]), cfg.snn_T, cfg.snn_v_init,\n                                  gains, cfg.snn_feat_scale, beta=cfg.snn_beta)\n        self.encoder_out_size = self.trunk.out_size\n\n    def forward(self, obs_dict):\n        return self.trunk(obs_dict["obs"])\n\n    def get_out_size(self):\n        return self.encoder_out_size\n\n\ndef make_snn_encoder(cfg, obs_space):\n    return SpikingEncoder(cfg, obs_space)\n\n\n# ----------------------------------------------------------------------------------------\n# Data-dependent gain calibration (so the network does not start silent)\n# ----------------------------------------------------------------------------------------\n@torch.no_grad()\ndef calibrate_gains(frames, T=8, target_rate=0.15, v_init=0.5, iters=18, device="cpu", seed=0, beta=0.9):\n    """Pick fixed per-layer current gains so each layer\'s mean firing rate is ~target_rate at\n    initialisation, using real (already standardised) frames. Weights use the same init Sample\n    Factory applies (orthogonal, gain 1, zero bias)."""\n    torch.manual_seed(seed)\n    trunk = SpikingTrunk(frames.shape[1], tuple(frames.shape[2:]), T, v_init, beta=beta).to(device)\n    for m in (trunk.conv1, trunk.conv2, trunk.conv3, trunk.fc):\n        nn.init.orthogonal_(m.weight, gain=1.0)\n        nn.init.zeros_(m.bias)\n    x = frames.to(device)\n    gains = [1.0, 1.0, 1.0, 1.0]\n\n    def rate_of_layer(k, g):\n        gg = list(gains)\n        gg[k] = g\n        c1 = trunk.conv1(x) * gg[0]\n        v = [None] * 4\n        total, count = 0.0, 0\n        for _ in range(T):\n            v[0], s1 = lif_step(v[0], c1, v_init=v_init, beta=beta)\n            if k == 0:\n                total += s1.sum().item(); count += s1.numel(); continue\n            v[1], s2 = lif_step(v[1], trunk.conv2(s1) * gg[1], v_init=v_init, beta=beta)\n            if k == 1:\n                total += s2.sum().item(); count += s2.numel(); continue\n            v[2], s3 = lif_step(v[2], trunk.conv3(s2) * gg[2], v_init=v_init, beta=beta)\n            if k == 2:\n                total += s3.sum().item(); count += s3.numel(); continue\n            v[3], s4 = lif_step(v[3], trunk.fc(s3.flatten(1)) * gg[3], v_init=v_init, beta=beta)\n            total += s4.sum().item(); count += s4.numel()\n        return total / count\n\n    for k in range(4):\n        lo, hi = -4.0, 4.0  # search log10(gain)\n        for _ in range(iters):\n            mid = 0.5 * (lo + hi)\n            if rate_of_layer(k, 10 ** mid) < target_rate:\n                lo = mid\n            else:\n                hi = mid\n        gains[k] = float(10 ** (0.5 * (lo + hi)))\n    rates = [rate_of_layer(k, gains[k]) for k in range(4)]\n    return gains, rates\n\n\n# ----------------------------------------------------------------------------------------\n# Standalone evaluator (no Sample Factory model), mirrors the PlainCNN / converted-SNN protocol\n# ----------------------------------------------------------------------------------------\nclass DirectSNN(nn.Module):\n    def __init__(self, T=8, v_init=0.5, gains=(1, 1, 1, 1), feat_scale=4.0, n_actions=4, beta=0.9):\n        super().__init__()\n        self.trunk = SpikingTrunk(3, (72, 128), T, v_init, gains, feat_scale, beta=beta)\n        self.head = nn.Linear(512, n_actions)\n\n    def forward(self, x, T=None, return_spikes=False):\n        out = self.trunk(x, T=T, return_spikes=return_spikes)\n        if return_spikes:\n            feats, spk = out\n            return self.head(feats), spk\n        return self.head(out)\n\n\ndef load_direct_snn(sf_state_dict, T, v_init=0.5, n_actions=4):\n    """Build the standalone SNN from a Sample Factory checkpoint[\'model\'] state dict."""\n    gains = sf_state_dict["encoder.trunk.gains"].cpu().tolist()\n    feat_scale = float(sf_state_dict["encoder.trunk.feat_scale"])\n    beta = float(sf_state_dict["encoder.trunk.beta"].cpu())\n    net = DirectSNN(T, v_init, gains, feat_scale, n_actions, beta=beta)\n    state = {}\n    for k, v in sf_state_dict.items():\n        if k.startswith("encoder.trunk."):\n            state["trunk." + k[len("encoder.trunk."):]] = v\n    state["head.weight"] = sf_state_dict["action_parameterization.distribution_linear.weight"]\n    state["head.bias"] = sf_state_dict["action_parameterization.distribution_linear.bias"]\n    missing, unexpected = net.load_state_dict(state, strict=False)\n    return net.eval(), missing, unexpected'
WORKER_SOURCE = 'import os, sys, json, random, hashlib, time, importlib\nfrom pathlib import Path\nfrom importlib.metadata import version\n# This variable also reaches newly spawned Sample Factory checkpoint loaders.\n# Only load checkpoints produced by this project or another trusted source.\nos.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"\nos.environ.setdefault("OMP_NUM_THREADS", "1")\nos.environ.setdefault("MKL_NUM_THREADS", "1")\nimport numpy as np\nimport torch\nimport gymnasium as gym\nROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]\n             if (p / "defend the center").is_dir() and (p / "conversion_friendly_v1").is_dir()), None)\nif ROOT is None:\n    raise RuntimeError("Start Jupyter from this repository.")\nWORK_DIR = ROOT / "defend the center"\nos.chdir(WORK_DIR)\nif str(WORK_DIR) not in sys.path: sys.path.insert(0, str(WORK_DIR))\nOUTPUT_ROOT = Path(os.environ.get("DEFEND_OUTPUT_ROOT", str(WORK_DIR / "outputs"))).expanduser().resolve()\nOUTPUT_ROOT.mkdir(parents=True, exist_ok=True)\ntorch.set_num_threads(1)\nassert torch.cuda.is_available(), "Launch this kernel inside a GPU allocation."\nDEVICE = "cuda"\nCPU_BUDGET = int(os.environ.get("SLURM_CPUS_PER_TASK", len(os.sched_getaffinity(0))))\nNUM_WORKERS = max(1, min(8, CPU_BUDGET, int(os.environ.get("DOOM_NUM_WORKERS", max(1, CPU_BUDGET-2)))))\nprint("Kernel:", sys.executable, "GPU:", torch.cuda.get_device_name(0), "workers:", NUM_WORKERS)\nprint("GPU test:", (torch.ones(1, device=DEVICE)+1).item())\n\nimport csv, re, subprocess, shutil\nfrom vizdoom import GameVariable\njob = json.loads(Path(sys.argv[1]).read_text())\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nSEED = job["seed"]\nENV_NAME = "doom_defend_the_center"\nEXPERIMENT = job["experiment"]\nRUN_MODE = job["mode"]\nTOTAL_STEPS = job["total_steps"]\nBATCH_SIZE = 1024; LEARNING_RATE = 1e-4; ENVS_PER_WORKER = 4\nSNN_T = 8; LIF_BETA = 0.9; TARGET_RATE = 0.15; FEAT_SCALE = 4.0\nrandom.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)\ntorch.cuda.manual_seed_all(SEED)\nimport functools\nfrom sample_factory.algo.utils.context import global_model_factory\nfrom sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args, load_from_checkpoint\nfrom sample_factory.envs.create_env import create_env\nfrom sample_factory.envs.env_utils import register_env\nfrom sample_factory.model.actor_critic import create_actor_critic\nfrom sample_factory.algo.utils.rl_utils import prepare_and_normalize_obs\nfrom sample_factory.train import run_rl\nfrom sf_examples.vizdoom.doom.doom_params import add_doom_env_args, doom_override_defaults\nfrom sf_examples.vizdoom.doom.doom_utils import DOOM_ENVS, make_doom_env_from_spec\n\nfor spec in DOOM_ENVS:\n    register_env(spec.name, functools.partial(make_doom_env_from_spec, spec))\nimport defend_lif as sd\nimportlib.reload(sd)\nglobal_model_factory().register_encoder_factory(sd.make_snn_encoder)\n\ndef parse_cfg(argv, evaluation=False):\n    parser, _ = parse_sf_args(argv=argv, evaluation=evaluation)\n    add_doom_env_args(parser); doom_override_defaults(parser)\n    sd.add_snn_args(parser)\n    return parse_full_cfg(parser, argv)\n\nTRAIN_DIR = str(OUTPUT_ROOT / "train_dir")\nRUN_DIR = Path(TRAIN_DIR) / EXPERIMENT\nif RUN_MODE == "fresh" and RUN_DIR.exists():\n    raise FileExistsError(f"Fresh run refuses to overwrite/resume {RUN_DIR}. Choose a new RUN_TAG.")\nif RUN_MODE != "fresh" and not (RUN_DIR / "config.json").is_file():\n    raise FileNotFoundError(f"No saved configuration in {RUN_DIR}")\nenv_cfg = parse_cfg([f"--env={ENV_NAME}", "--experiment=defend_probe",\n                     f"--train_dir={TRAIN_DIR}", f"--seed={SEED}",\n                     "--res_h=72", "--res_w=128", "--env_frameskip=4"], evaluation=True)\nenv = create_env(ENV_NAME, cfg=env_cfg, env_config=None)\nassert isinstance(env.action_space, gym.spaces.Tuple)\nassert [s.n for s in env.action_space.spaces] == [3, 2], env.action_space\nACTION_SPACE = env.action_space\nraw_space = env.observation_space\nOBS_SPACE = raw_space if isinstance(raw_space, gym.spaces.Dict) else gym.spaces.Dict({"obs": raw_space})\nassert OBS_SPACE["obs"].shape == (3, 72, 128)\nobs, _ = env.reset(seed=SEED)\nenv.step(ACTION_SPACE.sample())\nenv.close()\nprint("Observation:", OBS_SPACE, "Action:", ACTION_SPACE)\nprint("Turning: idle/left/right; attack: off/on. Five logits, six combinations.")\n\nif RUN_MODE != "evaluate":\n    if RUN_MODE == "fresh":\n        env = create_env(ENV_NAME, cfg=env_cfg, env_config=None)\n        env.action_space.seed(SEED)\n        raw_frames = []; episode = 0\n        try:\n            while len(raw_frames) < 800:\n                obs, _ = env.reset(seed=SEED + episode); episode += 1\n                done = truncated = False; t = 0\n                while not (done or truncated) and len(raw_frames) < 800:\n                    raw = obs["obs"] if isinstance(obs, dict) else obs\n                    if t % 2 == 0: raw_frames.append(np.asarray(raw).copy())\n                    obs, _, done, truncated, _ = env.step(env.action_space.sample()); t += 1\n        finally:\n            env.close()\n        x = torch.tensor(np.stack(raw_frames)).float() / 255.0\n        x = ((x-x.mean(0, keepdim=True))/torch.sqrt(x.var(0, unbiased=False, keepdim=True)+1e-5)).clamp(-5,5)\n        gains, rates = sd.calibrate_gains(x, T=SNN_T, target_rate=TARGET_RATE,\n                                         device=DEVICE, seed=SEED, beta=LIF_BETA)\n        path = OUTPUT_ROOT / "calibration" / EXPERIMENT\n        path.mkdir(parents=True, exist_ok=True)\n        torch.save({"frames": x, "gains": gains, "rates": rates, "seed": SEED,\n                    "beta": LIF_BETA, "T": SNN_T}, path / "initialization.pt")\n        print("Gains:", gains, "rates:", rates)\n    else:\n        # Never recalibrate a resumed model: read its original settings.\n        saved = json.loads((RUN_DIR / "config.json").read_text())\n        assert saved["env"] == ENV_NAME\n        gains = [float(v) for v in saved["snn_gains"].split(",")]\n        SNN_T = int(saved["snn_T"]); LIF_BETA = float(saved["snn_beta"])\n        FEAT_SCALE = float(saved["snn_feat_scale"])\n        print("Using saved LIF settings and gains.")\n    \n    argv = [f"--env={ENV_NAME}", f"--experiment={EXPERIMENT}", f"--train_dir={TRAIN_DIR}",\n            f"--seed={SEED}", "--device=gpu", "--use_rnn=False", "--nonlinearity=relu",\n            "--actor_critic_share_weights=True", "--res_h=72", "--res_w=128", "--env_frameskip=4",\n            f"--num_workers={NUM_WORKERS}", f"--num_envs_per_worker={ENVS_PER_WORKER}",\n            f"--batch_size={BATCH_SIZE}", f"--learning_rate={LEARNING_RATE}",\n            f"--train_for_env_steps={TOTAL_STEPS}", "--set_workers_cpu_affinity=False"]\n    argv += [f"--snn_T={SNN_T}", f"--snn_beta={LIF_BETA}",\n             "--snn_gains=" + ",".join(f"{g:.6f}" for g in gains), f"--snn_feat_scale={FEAT_SCALE}"]\n    if RUN_MODE != "evaluate":\n        cfg = parse_cfg(argv)\n        cfg.train_for_env_steps = TOTAL_STEPS\n        if RUN_MODE == "fresh" and RUN_DIR.exists():\n            raise FileExistsError("Run directory appeared; use a new tag or deliberately select resume.")\n        t0 = time.time()\n        status = run_rl(cfg)\n        print("Status:", status, "minutes:", (time.time()-t0)/60)\n        assert status == 0, "Training failed; inspect logs before evaluation."\n    else:\n        print("Training skipped: evaluation mode")\n\n    print("Training complete; run the evaluation cell next.")\n    sys.exit(0)\n\nreport_dir = Path(job["report_dir"])\nreport_dir.mkdir(parents=True,exist_ok=True)\ncheckpoints = sorted((RUN_DIR / "checkpoint_p0").glob("checkpoint_*.pth"))\nif not checkpoints: raise FileNotFoundError(f"No checkpoints in {RUN_DIR}")\nif job["kind"] == "latest":\n    CKPT_PATH = checkpoints[-1]\nelse:\n    candidates = list((RUN_DIR / "checkpoint_p0").glob("best_*.pth"))\n    if not candidates: raise FileNotFoundError("No best checkpoint available")\n    def rank(p):\n        m = re.search(r"_reward_([-+0-9.eE]+)\\.pth$",p.name)\n        if not m: raise ValueError(f"Unrecognized best-checkpoint name: {p.name}")\n        return float(m.group(1)),p.name\n    CKPT_PATH = max(candidates,key=rank)\nckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)\neval_cfg = parse_cfg([f"--env={ENV_NAME}", f"--experiment={EXPERIMENT}", f"--train_dir={TRAIN_DIR}"], evaluation=True)\neval_cfg = load_from_checkpoint(eval_cfg)\nassert eval_cfg.env == ENV_NAME and not eval_cfg.use_rnn\nac = create_actor_critic(eval_cfg, OBS_SPACE, ACTION_SPACE)\nac.load_state_dict(ckpt["model"], strict=True)\nac.model_to_device(torch.device(DEVICE)); ac.eval()\n\ndef policy_logits(raw_obs):\n    raw_dict = raw_obs if isinstance(raw_obs, dict) else {"obs": raw_obs}\n    batch = {k: torch.as_tensor(v).unsqueeze(0) for k,v in raw_dict.items()}\n    with torch.no_grad():\n        normalized = prepare_and_normalize_obs(ac, batch)\n        head = ac.forward_head(normalized)\n        core, _ = ac.forward_core(head, None)\n        return ac.forward_tail(core, values_only=False, sample_actions=False)["action_logits"]\n\ndef tuple_action(logits):\n    assert logits.shape == (1, 5)\n    return (int(logits[0,:3].argmax()), int(logits[0,3:].argmax()))\n\n# Check both factors independently, including turn-and-shoot combinations.\nassert tuple_action(torch.tensor([[0.,2.,1.,0.,3.]])) == (1,1)\nprint("Loaded:", CKPT_PATH.name, "steps:", ckpt["env_steps"])\n\n\ndef stats(game):\n    return {"health":float(game.get_game_variable(GameVariable.HEALTH)),\n            "ammo":float(game.get_game_variable(GameVariable.AMMO2)),\n            "kills":float(game.get_game_variable(GameVariable.KILLCOUNT))}\n\ndef play(seed, video=None):\n    env = create_env(ENV_NAME,cfg=eval_cfg,env_config=None,\n                     render_mode="rgb_array" if video else None)\n    process = None; video_log = None; records = []\n    total = used = kills = 0.; decision = 0\n    try:\n        obs,_ = env.reset(seed=seed); game=env.unwrapped.game\n        done=truncated=False\n        while not (done or truncated):\n            if video:\n                frame=env.render()\n                if frame is None: raise RuntimeError("No RGB frame returned")\n                frame=np.asarray(frame)\n                if frame.shape[0]==3 and frame.shape[-1]!=3: frame=frame.transpose(1,2,0)\n                if frame.ndim!=3 or frame.shape[-1]!=3: raise ValueError(frame.shape)\n                frame=np.ascontiguousarray(frame,dtype=np.uint8)\n                if process is None:\n                    h,w=frame.shape[:2];video_log=video.with_suffix(".ffmpeg.log").open("wb")\n                    process=subprocess.Popen([job["ffmpeg"],"-y","-f","rawvideo","-pix_fmt","rgb24",\n                        "-s",f"{w}x{h}","-r",str(35/eval_cfg.env_frameskip),"-i","pipe:0",\n                        "-an","-vf","pad=ceil(iw/2)*2:ceil(ih/2)*2","-c:v","libx264",\n                        "-pix_fmt","yuv420p","-movflags","+faststart",str(video)],\n                        stdin=subprocess.PIPE,stdout=subprocess.DEVNULL,stderr=video_log)\n                process.stdin.write(frame.tobytes())\n            before=stats(game);turn,attack=tuple_action(policy_logits(obs))\n            obs,reward,done,truncated,_=env.step((turn,attack));after=stats(game)\n            rounds_delta=max(0,before["ammo"]-after["ammo"])\n            kills_delta=max(0,after["kills"]-before["kills"])\n            used+=rounds_delta;kills+=kills_delta;total+=float(reward)\n            records.append({"seed":seed,"decision":decision,"turn":turn,"attack":attack,\n                "health_before":before["health"],"health_after":after["health"],\n                "ammo_before":before["ammo"],"ammo_after":after["ammo"],\n                "rounds_used":rounds_delta,"kills":kills_delta,"reward":float(reward)})\n            decision+=1\n        return {"seed":seed,"return":total,"kills":kills,"rounds_used":used,\n                "kills_per_round":kills/used if used else None,"decisions":decision},records\n    finally:\n        try:\n            if process:\n                try: process.stdin.close()\n                finally:\n                    try: process.wait(timeout=60)\n                    except subprocess.TimeoutExpired:\n                        process.kill();process.wait();raise RuntimeError("ffmpeg timed out")\n                if process.returncode: raise RuntimeError("ffmpeg failed; inspect its log")\n        finally:\n            if video_log: video_log.close()\n            env.close()\n\ndef csv_write(path,rows):\n    if not rows:return\n    with path.open("w",newline="") as f:\n        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)\n\nepisodes=[]\nwith (report_dir/"actions.csv").open("w",newline="") as action_file:\n    writer=None\n    for seed in job["eval_seeds"]:\n        row,actions=play(seed);episodes.append(row)\n        if writer is None:\n            writer=csv.DictWriter(action_file,fieldnames=list(actions[0]));writer.writeheader()\n        writer.writerows(actions);action_file.flush()\n        csv_write(report_dir/"episodes.csv",episodes)\n        print(f"seed={seed} return={row[\'return\']:.1f} kills={row[\'kills\']:.0f} rounds={row[\'rounds_used\']:.0f} kills/round={row[\'kills_per_round\']}",flush=True)\nrewards=np.array([r["return"] for r in episodes])\nrounds_total=sum(r["rounds_used"] for r in episodes)\nwith CKPT_PATH.open("rb") as f:checkpoint_hash=hashlib.file_digest(f,"sha256").hexdigest()\nsummary={"training_seed":SEED,"experiment":EXPERIMENT,"kind":job["kind"],\n    "checkpoint":str(CKPT_PATH),"checkpoint_sha256":checkpoint_hash,"env_steps":int(ckpt["env_steps"]),\n    "mean_return":float(rewards.mean()),"std_return":float(rewards.std()),\n    "SE":float(rewards.std()/np.sqrt(len(rewards))),\n    "mean_decisions":float(np.mean([r["decisions"] for r in episodes])),\n    "total_kills":sum(r["kills"] for r in episodes),"total_rounds":rounds_total,\n    "kills_per_round":sum(r["kills"] for r in episodes)/rounds_total if rounds_total else None,\n    "eval_seeds":job["eval_seeds"],"packages":{p:version(p) for p in ["torch","sample-factory","vizdoom","numpy","gymnasium"]},\n    "saved_config":json.loads((RUN_DIR/"config.json").read_text()),"python":sys.version,\n    "gpu":torch.cuda.get_device_name(0)}\n(report_dir/"metrics.json").write_text(json.dumps(summary,indent=2))\nprint("SUMMARY",json.dumps({k:v for k,v in summary.items() if k not in ["saved_config","eval_seeds","packages"]}),flush=True)\nvideo_records=[]\nfor seed in job["video_seeds"]:\n    path=report_dir/f"replay_seed{seed}.mp4"\n    row,actions=play(seed,path);row["video"]=str(path);video_records.append(row)\n    csv_write(report_dir/f"replay_seed{seed}_actions.csv",actions)\n(report_dir/"videos.json").write_text(json.dumps(video_records,indent=2))\n(report_dir/"complete.json").write_text(json.dumps({"completed":True}))\n'
(AUDIT_DIR / "defend_lif.py").write_text(MODULE_SOURCE)
WORKER_PATH = AUDIT_DIR / "audit_worker.py"
WORKER_PATH.write_text(WORKER_SOURCE)
ffmpeg_path = shutil.which("ffmpeg")
if VIDEO_SEEDS and not ffmpeg_path: raise RuntimeError("Load ffmpeg into PATH before recording videos")

def run_job(job, label):
    job_path = AUDIT_DIR / f"{label}_job.json"
    # Preserve the exact submitted settings rather than silently reusing different requests.
    if job_path.exists() and json.loads(job_path.read_text()) != job:
        raise RuntimeError(f"Settings changed for {label}. Choose a new AUDIT_TAG.")
    job_path.write_text(json.dumps(job,indent=2))
    log_path = AUDIT_DIR / f"{label}.log"
    process = subprocess.Popen([sys.executable,"-u",str(WORKER_PATH),str(job_path)],
        cwd=WORK_DIR,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    messages=queue.Queue()
    def drain():
        for line in process.stdout: messages.put(line)
        messages.put(None)
    threading.Thread(target=drain,daemon=True).start()
    try:
        with log_path.open("w") as log:
            while True:
                try: line=messages.get(timeout=30)
                except queue.Empty:
                    print(label,"still running; log:",log_path,flush=True);continue
                if line is None:break
                log.write(line);log.flush()
                if any(k in line for k in ["seed=","SUMMARY","Avg episode reward","Total num frames","Traceback","Error","Training complete"]):
                    print(line.rstrip(),flush=True)
        status=process.wait()
        if status:raise RuntimeError(f"{label} failed (exit {status}); inspect {log_path}")
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=20)
            except subprocess.TimeoutExpired:process.kill();process.wait()
        raise
    finally:
        process.stdout.close()
    return log_path


## Optional training

Default is skip. Reusing existing runs is the priority. Fresh training needs unused experiment names; resume preserves saved gains. If interrupted, inspect server worker processes before launching another training job. Finishing a retraining job does not remove earlier findings; retain those reports.

In [ ]:
if TRAINING_ACTION == "none":
    print("Training skipped; existing checkpoints will be evaluated.")
else:
    for seed in TRAINING_SEEDS:
        run_job({"seed":seed,"experiment":EXPERIMENTS[seed],"mode":TRAINING_ACTION,
                 "total_steps":TOTAL_TRAINING_STEPS},f"train_seed{seed}")


## Evaluate all checkpoints and record video

Prints per-episode ammo statistics; complete per-decision action/health/ammo logs are written to CSV. Each video has a matching action CSV, where decision 0 corresponds to the first frame. Inference uses the saved T and beta. No reward shaping or retraining occurs here.

In [ ]:
for seed in TRAINING_SEEDS:
    for kind in CHECKPOINT_KINDS:
        label=f"seed{seed}_{kind}"
        destination=AUDIT_DIR/label
        job={"seed":seed,"experiment":EXPERIMENTS[seed],"mode":"evaluate",
             "total_steps":TOTAL_TRAINING_STEPS,"kind":kind,"report_dir":str(destination),
             "eval_seeds":EVAL_SEEDS,"video_seeds":VIDEO_SEEDS,"ffmpeg":ffmpeg_path}
        previous=AUDIT_DIR/f"{label}_job.json"
        if previous.exists() and json.loads(previous.read_text()) != job:
            raise RuntimeError("Use a new AUDIT_TAG after changing evaluation settings")
        if (destination/"complete.json").exists():
            print("Reusing completed audit:",label);continue
        if destination.exists() and not RETRY_INCOMPLETE:
            raise RuntimeError(f"Incomplete {label}; inspect its log, then explicitly set RETRY_INCOMPLETE=True")
        run_job(job,label)


## Combined findings

Episode std/SE and across-training-seed variability answer different questions. Do not pool all episodes and call that training-seed uncertainty. With only three runs the across-seed estimate is preliminary. Keep latest and best results separate. Recorded checkpoints and settings are retained for auditing.

In [ ]:
import csv
summaries=[]
for seed in TRAINING_SEEDS:
    for kind in CHECKPOINT_KINDS:
        path=AUDIT_DIR/f"seed{seed}_{kind}"/"metrics.json"
        if not path.exists(): raise FileNotFoundError(path)
        summaries.append(json.loads(path.read_text()))
columns=["training_seed","kind","env_steps","mean_return","std_return","SE","mean_decisions","total_kills","total_rounds","kills_per_round","checkpoint_sha256"]
with (AUDIT_DIR/"comparison.csv").open("w",newline="") as f:
    writer=csv.DictWriter(f,fieldnames=columns);writer.writeheader()
    for row in summaries:writer.writerow({k:row[k] for k in columns})
print(f"{'Seed':>4} {'Kind':>7} {'Return':>9} {'Std':>8} {'Kills/round':>12} {'Decisions':>10}")
for row in summaries:
    efficiency=row["kills_per_round"]
    efficiency_text=f"{efficiency:.3f}" if efficiency is not None else "N/A"
    print(f"{row['training_seed']:4d} {row['kind']:>7s} {row['mean_return']:9.3f} {row['std_return']:8.3f} {efficiency_text:>12s} {row['mean_decisions']:10.2f}")
for kind in CHECKPOINT_KINDS:
    means=np.array([r["mean_return"] for r in summaries if r["kind"]==kind])
    spread=float(means.std(ddof=1)) if len(means)>1 else None
    print(kind,"mean of training-run means:",means.mean(),"sample std across training seeds:",spread)
# Flag config differences for review; seed, gain and path differences are expected.
configs={s:next(r["saved_config"] for r in summaries if r["training_seed"]==s) for s in TRAINING_SEEDS}
keys=["env","train_for_env_steps","learning_rate","batch_size","num_workers","num_envs_per_worker","env_frameskip","res_h","res_w","snn_T","snn_beta","snn_gains","snn_feat_scale","seed"]
for key in keys:
    values={s:configs[s].get(key) for s in TRAINING_SEEDS}
    if len({json.dumps(v,sort_keys=True) for v in values.values()})>1: print("CONFIG DIFFERENCE",key,values)
print("Saved:",AUDIT_DIR/"comparison.csv")


In [ ]:
# Display every recorded replay. Reduce VIDEO_SEEDS above if you want fewer files.
for seed in TRAINING_SEEDS:
    for kind in CHECKPOINT_KINDS:
        folder=AUDIT_DIR/f"seed{seed}_{kind}"
        print("Training seed:",seed,"checkpoint:",kind)
        for row in json.loads((folder/"videos.json").read_text()):
            print("Episode seed:",row["seed"],"return:",row["return"],"kills/round:",row["kills_per_round"])
            display(Video(row["video"],embed=True,width=640))


## Limits

Best training reward is not independent validation selection. This audit uses validation and demo
seeds that have already been examined. Kills per round is ammunition efficiency, not exact hit
accuracy. An action requesting attack may consume no ammo because of cooldown or empty ammo.
Rendering/logging can slow evaluation but does not train the policy. The same episode seed aligns
initial randomness; different actions produce different trajectories. Latest checkpoints can change
if training continues; use a new audit tag after further training, rather than reusing old reports.
This notebook does not add an ammo penalty, change frame skip, or average model weights.
